In [64]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
from src.parsers import parse_stories


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [65]:
df=pd.read_csv("../data/interim/listings_parsed.csv")

In [66]:
df.shape

(183490, 22)

In [67]:
df.head()

,Listing Title,Area,Distances to Railway Stations,Building Age,Stories Tall,Floor,Price,Administrative Costs,Deposit,Gratuity,...,Links to Listing,jnc,Price_yen,Administrative Costs_yen,Deposit_yen,Gratuity_yen,Admin_yen,deposit_months,gratuity_months,lookalike_id
0,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000055105229/?bc=...,55105229,65000,3000.0,NaN,NaN,3000.0,NaN,NaN,72960
1,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000055105230/?bc=...,55105230,65000,3000.0,NaN,NaN,3000.0,NaN,NaN,72960
2,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.8万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000054648287/?bc=...,54648287,68000,3000.0,NaN,NaN,3000.0,NaN,NaN,72961
3,カーザ コリーナ,東京都青梅市野上町２,ＪＲ青梅線/河辺駅 歩15分,築5年,3階建,3階,8.5万円,4000円,8.5万円,4.25万円,...,https://suumo.jp/chintai/jnc_000055414464/?bc=...,55414464,85000,4000.0,85000.0,42500.0,4000.0,1.0,0.5,20237
4,京王相模原線 京王よみうりランド駅 2階建 築14年,東京都稲城市矢野口,"京王相模原線/京王よみうりランド駅 歩10分,ＪＲ南武線/矢野口駅 歩17分,京王相模原線/...",築14年,2階建,1階,8.1万円,3900円,-,-,...,https://suumo.jp/chintai/jnc_000037983769/?bc=...,37983769,81000,3900.0,NaN,NaN,3900.0,NaN,NaN,84797


In [68]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id'],
      dtype='str')

In [69]:
df[["stories_above", "stories_below"]] = df["Stories Tall"].apply(lambda t: pd.Series(parse_stories(t)))
print((df["stories_above"] == "FAILED").sum())

213


In [70]:
df[df["Stories Tall"] == "平屋"][["Price_yen", "Square Meters", "Rooms"]].describe()

,Price_yen
count,213.000000
mean,96704.225352
std,56725.687428
min,39000.000000
25%,65000.000000
50%,80000.000000
75%,100000.000000
max,350000.000000


In [71]:
df.loc[df["stories_above"] == "FAILED", "Stories Tall"].value_counts().head(10)

Stories Tall
平屋    213
Name: count, dtype: int64

In [72]:
df['Floor'].value_counts().head(50) 

Floor
1階       60974
2階       54627
3階       26766
4階       12926
5階        7333
6階        4580
7階        3177
8階        2439
1-2階      2054
9階        1765
10階       1546
-          997
11階        857
B1階        479
13階        454
12階        418
1-3階       270
14階        258
15階        180
21階        143
19階        105
2-3階       104
16階         88
23階         74
24階         64
30階         61
18階         55
25階         46
29階         46
B2階         45
3-4階        42
17階         42
20階         41
35階         38
1-1階        38
B1-1階       36
22階         28
4-5階        24
2-1階        21
28階         17
32階         17
B1-2階       16
51階         16
33階         14
1-階         12
2-0階        11
27階          8
1-4階         8
1-10階        8
34階          7
Name: count, dtype: int64

In [73]:
df["Floor"].str.replace(r"\d", "9", regex=True).value_counts()

Floor
9階        174587
99階         4652
9-9階        2617
-            997
B9階          529
B9-9階         55
9-階           27
9-99階          9
M9階            5
99-階           4
99-99階         3
99-9階          2
9B9階           2
B9-階           1
Name: count, dtype: int64

In [74]:
from src.parsers import parse_floor
for t in ["1階", "12階", "1-2階", "1-階", "B1階", "B1-1階", "-", "M1階", "1B1階"]:
    print(repr(t), parse_floor(t))

'1階' (1, 1, 0)
'12階' (12, 12, 0)
'1-2階' (1, 2, 1)
'1-階' (1, 1, 1)
'B1階' (-1, -1, 0)
'B1-1階' (-1, 1, 1)
'-' (None, None, None)
'M1階' ('FAILED', None, None)
'1B1階' ('FAILED', None, None)


In [75]:
df[["floor_min", "floor_max", "is_multi_floor"]] = df["Floor"].apply(lambda t: pd.Series(parse_floor(t)))
print((df["floor_min"] == "FAILED").sum())

7


In [76]:
df.loc[df["floor_min"] == "FAILED", "Floor"].value_counts()

Floor
M2階     5
1B2階    2
Name: count, dtype: int64

In [77]:
for c in ["floor_min", "floor_max", "is_multi_floor"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df["is_multi_floor"].value_counts())
print(df[["floor_min", "floor_max"]].describe())
print(df["floor_min"].isna().sum())

is_multi_floor
0.0    179768
1.0      2718
Name: count, dtype: int64
           floor_min      floor_max
count  182486.000000  182486.000000
mean        2.690672       2.706525
std         2.643514       2.637635
min        -4.000000      -4.000000
25%         1.000000       1.000000
50%         2.000000       2.000000
75%         3.000000       3.000000
max        51.000000      51.000000
1004


In [78]:
df["Square Meters"].str.replace(r"\d", "9", regex=True).value_counts()

Square Meters
99.99m     128518
99.9m       29293
99m         23058
999.99m      1412
9.99m         550
999.9m        249
999m          183
9.9m          123
9m            104
Name: count, dtype: int64

In [79]:
df['Square Meters'].value_counts().head(50) 

Square Meters
19.87m    2044
20m       2037
18m       1487
40m       1155
21m       1116
30m        904
17m        894
25m        868
16m        831
22m        732
35m        686
23m        682
19m        665
25.5m      662
15m        606
24m        605
20.28m     600
33m        587
19.44m     567
17.41m     562
19.8m      526
25.2m      517
26m        496
23.18m     475
16.5m      463
25.37m     459
26.08m     458
25.52m     454
38m        447
25.65m     438
25.6m      422
45m        421
36m        413
17.39m     413
28m        409
25.92m     408
42m        397
17.01m     396
25.38m     370
25.55m     357
27m        357
20.25m     353
25.11m     342
22.68m     337
50m        333
32m        331
22.35m     317
34m        316
38.88m     313
19.83m     309
Name: count, dtype: int64

In [80]:
from src.parsers import parse_area
for t in ["19.87m", "20m", "abc"]:
    print(repr(t), parse_area(t))

'19.87m' 19.87
'20m' 20.0
'abc' ('FAILED', None)


In [81]:
df['Area'] = df['Square Meters'].apply(parse_area)
print((df["Area"] == "FAILED").sum())

0


In [82]:
df['Area'].describe()

count    183490.000000
mean         34.212695
std          19.580601
min           1.000000
25%          21.120000
50%          26.900000
75%          42.050000
max         562.630000
Name: Area, dtype: float64

In [83]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor'],
      dtype='str')

In [84]:
df.head()

,Listing Title,Area,Distances to Railway Stations,Building Age,Stories Tall,Floor,Price,Administrative Costs,Deposit,Gratuity,...,Gratuity_yen,Admin_yen,deposit_months,gratuity_months,lookalike_id,stories_above,stories_below,floor_min,floor_max,is_multi_floor
0,ルアナ,47.32,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,NaN,3000.0,NaN,NaN,72960,2,0.0,1.0,1.0,0.0
1,ルアナ,47.32,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,NaN,3000.0,NaN,NaN,72960,2,0.0,1.0,1.0,0.0
2,ルアナ,51.06,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.8万円,3000円,-,-,...,NaN,3000.0,NaN,NaN,72961,2,0.0,1.0,1.0,0.0
3,カーザ コリーナ,57.29,ＪＲ青梅線/河辺駅 歩15分,築5年,3階建,3階,8.5万円,4000円,8.5万円,4.25万円,...,42500.0,4000.0,1.0,0.5,20237,3,0.0,3.0,3.0,0.0
4,京王相模原線 京王よみうりランド駅 2階建 築14年,42.49,"京王相模原線/京王よみうりランド駅 歩10分,ＪＲ南武線/矢野口駅 歩17分,京王相模原線/...",築14年,2階建,1階,8.1万円,3900円,-,-,...,NaN,3900.0,NaN,NaN,84797,2,0.0,1.0,1.0,0.0


In [85]:
df['Building Age'].value_counts().head(50)

Building Age
新築      28963
築31年     6450
築32年     6231
築30年     5985
築29年     5750
築33年     5260
築28年     4997
築2年      4979
築14年     4944
築27年     4913
築3年      4897
築13年     4851
築15年     4457
築34年     4242
築17年     4208
築16年     4027
築26年     3805
築5年      3635
築12年     3557
築4年      3510
築35年     3431
築18年     3400
築11年     3149
築6年      2895
築22年     2735
築19年     2735
築25年     2728
築23年     2668
築36年     2628
築1年      2615
築21年     2580
築7年      2524
築20年     2476
築24年     2381
築9年      2233
築8年      2180
築10年     2139
築37年     1898
築40年     1512
築41年     1497
築38年     1441
築42年     1270
築39年     1137
築47年     1039
築43年     1037
築48年      907
築44年      894
築46年      750
築45年      722
築49年      688
Name: count, dtype: int64

In [86]:
# Number of buildings with 築1年
df[df['Building Age'] == "築1年"][["Price_yen", "Square Meters", "Rooms"]].describe()

,Price_yen
count,2615.000000
mean,129994.531549
std,62710.623745
min,42000.000000
25%,92500.000000
50%,115000.000000
75%,150000.000000
max,650000.000000


In [87]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor'],
      dtype='str')

In [88]:
from src.parsers import parse_building_age
df['Building_Age_Parsed']=df['Building Age'].apply(parse_building_age)
print((df['Building_Age_Parsed'] == "FAILED").sum())

0


In [89]:
df['Building_Age_Parsed'].value_counts()

Building_Age_Parsed
(0, 1)     28963
(31, 0)     6450
(32, 0)     6231
(30, 0)     5985
(29, 0)     5750
           ...  
(82, 0)        2
(69, 0)        2
(70, 0)        2
(74, 0)        1
(77, 0)        1
Name: count, Length: 82, dtype: int64

In [95]:
print((df["age_years"] == "FAILED").sum())   # run BEFORE to_numeric
print(df["is_new"].isna().sum())             # should be 4 if so 

4
4


In [91]:
df[["age_years", "is_new"]] = df["Building Age"].apply(lambda t: pd.Series(parse_building_age(t)))
print((df["age_years"] == "FAILED").sum())   # run BEFORE to_numeric
print(df["is_new"].isna().sum())             # should be 4 if so 



4
4


In [97]:
df["age_years"] = pd.to_numeric(df["age_years"], errors="coerce")
print(df["is_new"].value_counts())   

is_new
0.0    154523
1.0     28963
Name: count, dtype: int64


In [100]:
bad = df[df["is_new"].isna()]
for idx, row in bad.iterrows():
    print(idx, "|", row["Building Age"], "|", row["Stories Tall"], "|", row["Price"])

26818 | 築99年以上 | 2階建 | 10万円
48375 | 築99年以上 | 4階建 | 7.6万円
163818 | 築99年以上 | 2階建 | 3.3万円
181690 | 築99年以上 | 2階建 | 7.15万円


   - 4 rows failed the age parser (values: [list]). Rule: [what you chose]. Test added.

In [101]:
print(bad["Building Age"].map(repr).tolist())

["'築99年以上'", "'築99年以上'", "'築99年以上'", "'築99年以上'"]


In [103]:
df["age_is_lower_bound"] = df["Building Age"].str.contains("以上").astype(int)
print(df["age_is_lower_bound"].sum())   # expect 4

4


In [104]:
df[["age_years", "is_new"]] = df["Building Age"].apply(lambda t: pd.Series(parse_building_age(t)))
print((df["age_years"] == "FAILED").sum())      # expect 0
df["age_years"] = pd.to_numeric(df["age_years"], errors="coerce")
print(df["is_new"].value_counts())              # expect 154,527 and 28,963, total 183,490
print(df["age_years"].max())                    # now 99
print(df["age_is_lower_bound"].sum())           # expect 4

0
is_new
0    154527
1     28963
Name: count, dtype: int64
99
4


In [105]:
df.to_csv("../data/interim/listings_parsed_v2.csv", index=False, encoding="utf-8-sig")